In [ ]:
import os
# For a fair compiraison of performance against NumPy, we disable multithreading
os.environ["XLA_FLAGS"] = "--xla_cpu_multi_thread_eigen=false intra_op_parallelism_threads=1"
os.environ["OMP_NUM_THREADS"] = "1"

import jax

# For a fair comparaison of performance against NumPy, we use doubles for JAX computations
jax.config.update('jax_enable_x64', True)

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import time

# Randomness in JAX

NumPy hides its random generator: each call to `np.random.*` quietly reads internal state and advances it, so the next call differs.
That hidden mutation is a side effect, which already makes the design a poor fit for JAX's functional model.
However, the sharper problem is the lack of reproducibility with this design choice.
Once many draws run in parallel, as on a GPU or on multiprocessed CPU, the order in which they touch that shared state is no longer guaranteed, so the same program can sample differently from one run to the next.

To solve this issue and also match JAX functionnal (purity) design, JAX keeps no hidden generator state which needs to be managed manually.
Every random function instead takes an explicit key as its first argument, and its output is a pure function of that key.

JAX's random system is built on a [counter-based random number generator](https://en.wikipedia.org/wiki/Counter-based_random_number_generator) (CBRNG), which suits massively parallel computation. [It defaults to a Threefry CBRNG](https://docs.jax.dev/en/latest/jax.random.html#design-and-background), though [other generators are available](https://docs.jax.dev/en/latest/jax.random.html#advanced-rng-configuration) (which won't be explored in the following notebook).

> _**NOTE:**_ If you're interested in more details about the internals of this design choice, read this [link](https://github.com/jax-ml/jax/blob/main/docs/jep/263-prng.md) from the internal JAX documentation.

## Generating a key and first sampling

All random-related operations live in the `jax.random` submodule ([link to the documentation](https://docs.jax.dev/en/latest/jax.random.html), read it!).
To generate a key from a given seed, use `jax.random.key`:

In [ ]:
seed = 1337
key = jax.random.key(seed)
print(key)

Each random number samplers takes as first argument a key.
To sample from a normal distribution, use `jax.random.normal`:

> _**Note:**_ Unlike `np.random.normal`, which takes `loc` and `scale` arguments, `jax.random.normal` samples from a standard normal distribution, so rescaling is necessary: `loc + scale*jax.random.normal(...)`

In [ ]:
print(jax.random.normal(key))

Using the same key will always lead to the same numbers drawn

In [ ]:
print(jax.random.normal(key))
print(jax.random.normal(key))

To generate multiple samples, use the `shape` argument (which accepts a shape tuple or an integer):

In [ ]:
samples = jax.random.normal(key, shape=100_000)
plt.hist(samples, density=True, bins=100)
plt.show()

Not that for a fixed key, samples are fixed by their row-major position: `jax.random.normal(key, shape)` equals `jax.random.normal(key, (size,)).reshape(shape)`:

In [ ]:
print(jax.random.normal(key, shape=4))
print(jax.random.normal(key, shape=(2, 2)))

## Key splitting

To generate news keys from a given key, split it ([link to the documentation](https://docs.jax.dev/en/latest/_autosummary/jax.random.split.html), read it!):

In [ ]:
key, subkey = jax.random.split(key)
print(jax.random.normal(key))
print(jax.random.normal(subkey))

`split` can also generate an arbitrary number of keys using the `num` argument:

In [ ]:
keys = jax.random.split(key, num=1000)
print(len(keys))

Key splitting is a somewhat slow operation.
When you just need a block of samples, draw them all at once from one key rather than splitting into many and sampling separately.
Both give equally independent samples, but splitting does more work: it builds the keys first,
then samples from each, while the direct call produces the whole block in one pass.

This can be shown with the following benchmark.

In [ ]:
@jax.jit(static_argnames=('N', 'M'))
def sample_split(key, N, M):
    _sample = jax.vmap(lambda key: jax.random.normal(key, shape=M))
    return _sample(jax.random.split(key, num=N))

@jax.jit(static_argnames=('N', 'M'))
def sample_direct(key, N, M):
    return jax.random.normal(key, shape=(N, M))

In [ ]:
N, M = 100000, 10
# JIT + warmup
sample_split(key, N, M)
sample_direct(key, N, M)
print()

In [ ]:
%timeit sample_split(key, N, M)
%timeit sample_direct(key, N, M)

As it can be seen, it is faster to directly generate a large amount of samples in one go than in several iterations.

> _**Question:**_ Vary `N` and `M`. When do the two timings converge and pull apart ? Why?

# Random sampling performance

In this section we will discuss performances of the random samplers and compare them to NumPy.

Let us first define a basic benchmark function as `timeit` has a quite clunky interface when used programmatically.

In [ ]:
def bench(fn, *args, warmup=3, runs=100):
    for _ in range(warmup):
        fn(*args)
    times = np.empty(runs)
    
    for i in range(runs):
        t0 = time.perf_counter()
        fn(*args)
        times[i] = time.perf_counter() - t0
        
    return times.min()

## Inverse transform sampling
Some samplers rely on [inverse transform sampling](https://en.wikipedia.org/wiki/Inverse_transform_sampling): draw a uniform, then push it through the distribution's inverse CDF (quantile function).
Every such sampler is a uniform draw plus one elementwise transform, so their costs are all similar and comparable to NumPy.

Sampling from a Normal distribution for both JAX and NumPy uses inverse transform sampling.
In the following we compare performance of both Normal and uniform sampling for JAX and NumPy.

In [ ]:
@jax.jit(static_argnames='N')
def sample_normal_jax(N, key):
    return jax.random.normal(key, shape=N)

@jax.jit(static_argnames='N')
def sample_uniform_jax(N, key):
    return jax.random.uniform(key, shape=N)

def sample_normal_numpy(N):
    return np.random.normal(size=N)

def sample_uniform_numpy(N):
    return np.random.uniform(size=N)

In [ ]:
times = {'jax_normal': [],
         'jax_uniform': [],
         'np_normal': [],
         'np_uniform': []}

N = np.logspace(2, 6, 10, dtype=int)

runs = 500
for n in N:
    times['jax_normal'].append(bench(lambda _n, _key: sample_normal_jax(_n, _key).block_until_ready(), int(n), jax.random.key(1337), runs=runs))
    times['jax_uniform'].append(bench(lambda _n, _key: sample_uniform_jax(_n, _key).block_until_ready(), int(n), jax.random.key(1337), runs=runs))
    
    times['np_normal'].append(bench(sample_normal_numpy, n, runs=runs))
    times['np_uniform'].append(bench(sample_uniform_numpy, n, runs=runs))



In [ ]:
plt.plot(N, times['jax_normal'], label="JAX Normal")
plt.plot(N, times['jax_uniform'], label="JAX Uniform")

plt.plot(N, times['np_normal'], label="NumPy Normal")
plt.plot(N, times['np_uniform'], label="NumPy Uniform")

plt.xscale('log')
plt.yscale('log')
plt.xlabel("$N$")
plt.ylabel("Elapsed time [s]")
plt.legend()
plt.grid()
plt.show()

## Poisson sampling performance

Not every distribution has a cheap quantile function.
Such algorithms are usually split into regimes, with a different method tuned for each parameter range.
In most simpler cases, samplers uses [rejection](https://en.wikipedia.org/wiki/Rejection_sampling): propose a value, accept it or try again.
Rejection sampling has real drawbacks, and grows especially inefficient for highly inhomogeneous distributions or in high dimensions.

To illustrate this, we will take Poisson sampling as an example (you can see its implementation for JAX v0.11.0 [here](https://github.com/jax-ml/jax/blob/a1521744c6dc074443fe549f19f48d7197abf759/jax/_src/random/core.py#L1816)).
JAX's Poisson sampler splits at $\lambda = 10$ into two regimes:
- First, below it, [Knuth's method](https://en.wikipedia.org/wiki/Poisson_distribution#Evaluating_the_Poisson_distribution) multiplies uniforms until their product falls under a threshold, needing about $\lambda$ draws per sample.
This is cheap for small rates and gets steadily worse as the rate grows.
- Then, above this threshold, Hörmann's transformed rejection takes over, whose cost barely depends on $\lambda$ at all.

The switch is implemented as a `jax.lax.select`: both algorithms run over the whole array and one result is discarded, with the unused one fed dummy rates chosen so it terminates as quickly as possible.
This is a particularly inefficient scheme, but unavoidable under JAX's programming model.

On the other hand, this regime structure suits NumPy perfectly.
Its scalar loop lets every element allow its own branch and stop as soon as it is accepted, so the cost paid is the average one.

We now benchmark both implementations by sweeping $\lambda$ up to 100.

In [ ]:
@jax.jit(static_argnames='N')
def jax_poisson(key, lam, N):
    return jax.random.poisson(key, lam, shape=N)

def np_poisson(lam, N):
    return np.random.poisson(lam, size=N)

def benchmark_poisson(lambdas, N, runs):
    times = {'jax': [],
             'np': []}
    for lam in lambdas:
        times['jax'].append(bench(lambda: jax_poisson(key, lam, N).block_until_ready(), runs=runs))
        times['np'].append(bench(lambda: np_poisson(lam, N), runs=runs))
    return times

In [ ]:
# Warning, this will take some time
N = 100_000
lambdas = np.arange(1, 100).tolist()
times = benchmark_poisson(lambdas, N, 100)

In [ ]:
# Convert timings in mega samples per second
plt.plot(lambdas, N/np.array(times['jax'])/1e6, label="JAX")
plt.plot(lambdas, N/np.array(times['np'])/1e6, label="NumPy")
plt.axvline(10., ls='--', color='black')
plt.legend()
plt.xlabel("$\\lambda$")
plt.ylabel("MSPS")
plt.grid()
plt.show()

NumPy is at least twice as fast as JAX here, and three to four times on average.

> _**Note:**_ We are here comparing number of samples per second, meaning that higher is better/faster.

This penalty is less of a show-stopper than it looks.
Sampling is rarely the bottleneck of a real computation, and the cost is usually recovered downstream, once the sampler is fused into a larger jitted program and composed with other JAX transformations.
More fundamentally, this style of implementation is what accelerators demand: on a GPU there is no scalar loop to fall back on, and a data-parallel sampler is the only kind that runs at all.

> _**Note:**_ From reading the JAX source code, it seems that its latest version (not yet available on PyPu) implements an approximate loop-less method using some advanced trickery. I had not yet the opportunity to explore this as of today.

## Benchmark your favorite random distribution

Pick your favorite distribution or whichever one dominates your own work and benchmark `jax.random` against NumPy for it.
You may look at its implementation and predict its different regimes if it has any.